In [1]:
#Load libraries
import pandas as pd
from pathlib import Path

#File paths
input_file = Path("../data/online_retail_II.xlsx")
output_file = Path("../data/online_retail_II_cleaned.csv")

In [2]:
#Investigate the Excel file sheets  
excel_file = pd.ExcelFile(input_file)

print("==== EXCEL WORKBOOK INFORMATION ====")
print("Sheets found:",excel_file.sheet_names)


==== EXCEL WORKBOOK INFORMATION ====
Sheets found: ['Year 2009-2010', 'Year 2010-2011']


In [3]:
#Load all sheets into a dictionary
all_sheets = {}

for sheet in excel_file.sheet_names:

    print(f"\nLoading sheet: {sheet}")

    all_sheets[sheet] = pd.read_excel(
        input_file,
        sheet_name=sheet
    )

    print(
        f"Rows: {len(all_sheets[sheet]):,}"
    )

    print(
        f"Columns: {len(all_sheets[sheet].columns)}"
    )

print("\n==== SHEET SUMMARY ====")

for sheet, data in all_sheets.items():

    print(
        f"{sheet}: "
        f"{len(data):,} rows × "
        f"{len(data.columns)} columns"
    )




Loading sheet: Year 2009-2010
Rows: 525,461
Columns: 8

Loading sheet: Year 2010-2011
Rows: 541,910
Columns: 8

==== SHEET SUMMARY ====
Year 2009-2010: 525,461 rows × 8 columns
Year 2010-2011: 541,910 rows × 8 columns


In [4]:
#Check for column consistency across all sheets
print("\n==== COLUMN CONSISTENCY CHECK ====")


first_sheet = excel_file.sheet_names[0]

reference_columns = list(
    all_sheets[first_sheet].columns
)

for sheet, data in all_sheets.items():

    current_columns = list(data.columns)

    if current_columns == reference_columns:
        print(f"{sheet}: OK")
    else:
        print(f"{sheet}: DIFFERENT COLUMNS")

        print("Columns:")
        print(current_columns)



==== COLUMN CONSISTENCY CHECK ====
Year 2009-2010: OK
Year 2010-2011: OK


In [5]:
#COMBINE ALL SHEETS

df = pd.concat(
    all_sheets.values(),
    ignore_index=True
)


print("\n==== COMBINED DATASET ====")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nColumns:")
print(df.columns.tolist())




==== COMBINED DATASET ====
Rows: 1,067,371
Columns: 8

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [6]:
#Check th combined dataset first 5 rows and last 5 rows
print("\n===== FIRST 5 ROWS =====")
print(
    df.head().to_string(index=False)
)

print("\n===== LAST 5 ROWS =====")
print(
    df.tail().to_string(index=False)
)


===== FIRST 5 ROWS =====
Invoice StockCode                         Description  Quantity         InvoiceDate  Price  Customer ID        Country
 489434     85048 15CM CHRISTMAS GLASS BALL 20 LIGHTS        12 2009-12-01 07:45:00   6.95      13085.0 United Kingdom
 489434    79323P                  PINK CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0 United Kingdom
 489434    79323W                 WHITE CHERRY LIGHTS        12 2009-12-01 07:45:00   6.75      13085.0 United Kingdom
 489434     22041        RECORD FRAME 7" SINGLE SIZE         48 2009-12-01 07:45:00   2.10      13085.0 United Kingdom
 489434     21232      STRAWBERRY CERAMIC TRINKET BOX        24 2009-12-01 07:45:00   1.25      13085.0 United Kingdom

===== LAST 5 ROWS =====
Invoice StockCode                     Description  Quantity         InvoiceDate  Price  Customer ID Country
 581587     22899    CHILDREN'S APRON DOLLY GIRL          6 2011-12-09 12:50:00   2.10      12680.0  France
 581587     23254  

In [7]:

#Check missing values across all sheets
print("\n==== MISSING VALUES ====")
missing_values = df.isna().sum()
print(missing_values)



==== MISSING VALUES ====
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64


In [9]:
#Check for duplicate rows in the combined dataset
duplicate_count = df.duplicated().sum()


print("\n==== DUPLICATE CHECK ====")
print(f"Exact duplicate rows: {duplicate_count:,}")





==== DUPLICATE CHECK ====
Exact duplicate rows: 34,335


In [14]:
#Remove cancellations

cancellation_mask = (
    df["Invoice"]
    .astype(str)
    .str.startswith("C")
)

cancellations_removed = (
    cancellation_mask.sum()
)

df = df[~cancellation_mask]

print("\n==== CANCELLATION REMOVAL ====")
print(f"Rows removed due to cancellations: {cancellations_removed:,}")


==== CANCELLATION REMOVAL ====
Rows removed due to cancellations: 0


In [15]:
#Remove non-positive quantities
quantity_mask = df["Quantity"] <= 0

quantity_rows_removed = (
    quantity_mask.sum()
)

df = df[~quantity_mask]

print("\n==== NON-POSITIVE QUANTITY REMOVAL ====")
print(f"Rows removed due to non-positive quantities: {quantity_rows_removed:,}")


==== NON-POSITIVE QUANTITY REMOVAL ====
Rows removed due to non-positive quantities: 0


In [17]:
#Remove non-positive prices
price_mask = df["Price"] <= 0

price_rows_removed = (
    price_mask.sum()
)

df = df[~price_mask]

print("\n==== NON-POSITIVE PRICE REMOVAL ====")
print(f"Rows removed due to non-positive prices: {price_rows_removed:,}")


==== NON-POSITIVE PRICE REMOVAL ====
Rows removed due to non-positive prices: 2,750


In [18]:
#Create Revenue 
df["Revenue"] = (
    df["Quantity"] * df["Price"]
)

df.head() 


/var/folders/g_/krz5pkj97b57ldkhdmpmxqlm0000gn/T/ipykernel_5278/262320331.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Revenue"] = (


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [21]:
#Standardize data types
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"]
)

df["Quantity"] = (
    df["Quantity"].astype(int)
)

df["Price"] = (
    df["Price"].astype(float)
)

df["Revenue"] = (
    df["Revenue"].astype(float)
)

df.head()

/var/folders/g_/krz5pkj97b57ldkhdmpmxqlm0000gn/T/ipykernel_5278/1296185132.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["InvoiceDate"] = pd.to_datetime(
/var/folders/g_/krz5pkj97b57ldkhdmpmxqlm0000gn/T/ipykernel_5278/1296185132.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Quantity"] = (
/var/folders/g_/krz5pkj97b57ldkhdmpmxqlm0000gn/T/ipykernel_5278/1296185132.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,co

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [24]:
#Sort data by InvoiceDate and InvoiceNo
df = df.sort_values(
    by=["InvoiceDate", "Invoice"]
).reset_index(drop=True)

print("\n==== SORTED DATASET ====")
print(f"Rows: {len(df):,}")


==== SORTED DATASET ====
Rows: 1,041,670


In [25]:
#Save the cleaned dataset to a CSV file 
df.to_csv(
    output_file,
    index=False
)


In [35]:
#Final summary of the cleaning process


print("\n")
print("=" * 70)
print("FINAL CLEAN DATASET")
print("=" * 70)

print(f"Original rows:  {len(df):,}")
print(f"Duplicates removed: {duplicate_count:,}" )

print(
    f"Cancellations removed: "
    f"{cancellations_removed:,}"
)

print(
    f"Non-positive quantities removed: "
    f"{quantity_rows_removed:,}"
)

print(
    f"Non-positive prices removed: "
    f"{price_rows_removed:,}"
)

print(
    f"Final rows: "
    f"{len(df):,}"
)

print(
    f"Duplicate rows remaining: "
    f"{df.duplicated().sum():,}"
)

print(
    f"Negative quantities remaining: "
    f"{(df['Quantity'] < 0).sum():,}"
)

print(
    f"Non-positive prices remaining: "
    f"{(df['Price'] <= 0).sum():,}"
)

print(
  f"Missing CustomerID remaining: "
  f"{df['Customer ID'].isna().sum():,}"
)


print(
    f"Unique customers: "
    f"{df['Customer ID'].nunique():,}"
)

print(
    f"Unique products: "
    f"{df['StockCode'].nunique():,}"
)

print(
    f"Unique countries: "
    f"{df['Country'].nunique():,}"
)

print(
    f"Date range: "
    f"{df['InvoiceDate'].min()} "
    f"to "
    f"{df['InvoiceDate'].max()}"
)

print(
    f"Total revenue: "
    f"£{df['Revenue'].sum():,.2f}"
)

print("=" * 70)

print(
    f"\nClean dataset saved to: "
    f"{output_file}"
)



FINAL CLEAN DATASET
Original rows:  1,041,670
Duplicates removed: 34,335
Cancellations removed: 0
Non-positive quantities removed: 0
Non-positive prices removed: 2,750
Final rows: 1,041,670
Duplicate rows remaining: 33,757
Negative quantities remaining: 0
Non-positive prices remaining: 0
Missing CustomerID remaining: 236,121
Unique customers: 5,878
Unique products: 4,917
Unique countries: 43
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Total revenue: £20,972,594.57

Clean dataset saved to: ../data/online_retail_II_cleaned.csv
